In [3]:
!python -m venv myenv 
!./myenv/bin/pip install ipykernel
!./myenv/bin/python -m ipykernel install --user --name=myenv --display-name="Python (myenv)"

Installed kernelspec myenv in /home/murgi/.var/app/com.visualstudio.code/data/jupyter/kernels/myenv


In [8]:
%env TMPDIR=/home/user/Documents/pip_tmp
!./myenv/bin/pip install sentence-transformers faiss-cpu pandas numpy tqdm

env: TMPDIR=/home/user/Documents/pip_tmp


In [1]:
import pickle
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


/home/murgi/Documents/rag_gbm/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CGGA_693_PATH = "CGGA.mRNAseq_693_clinical.20200506.txt"

df = pd.read_csv(
    CGGA_693_PATH,
    sep="\t"
)

print(df.shape)
df.head()

(693, 13)


,CGGA_ID,PRS_type,Histology,Grade,Gender,Age,OS,Censor (alive=0; dead=1),Radio_status (treated=1;un-treated=0),Chemo_status (TMZ treated=1;un-treated=0),IDH_mutation_status,1p19q_codeletion_status,MGMTp_methylation_status
0,CGGA_1002,Primary,AA,WHO III,Female,43.0,305.0,1.0,1.0,1.0,Wildtype,Non-codel,methylated
1,CGGA_1003,Primary,O,WHO II,Female,47.0,3817.0,0.0,0.0,1.0,Mutant,Codel,un-methylated
2,CGGA_1010,Primary,A,WHO II,Male,45.0,246.0,1.0,1.0,1.0,Mutant,NaN,un-methylated
3,CGGA_1012,Recurrent,rO,WHO II,Male,45.0,3679.0,1.0,1.0,1.0,Mutant,Non-codel,un-methylated
4,CGGA_1014,Primary,A,WHO II,Male,42.0,263.0,1.0,0.0,1.0,Wildtype,Non-codel,un-methylated


In [3]:
df = df.fillna("Unknown")

print("Total patients:", len(df))

Total patients: 693


In [4]:
documents = []
metadata = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    doc = f"""
Patient ID {row['CGGA_ID']}.

{row['PRS_type']} {row['Histology']}.
{row['Grade']}.

Gender: {row['Gender']}.
Age: {row['Age']} years.

IDH mutation status:
{row['IDH_mutation_status']}.

1p19q codeletion status:
{row['1p19q_codeletion_status']}.

MGMT methylation status:
{row['MGMTp_methylation_status']}.

Radiotherapy:
{row['Radio_status (treated=1;un-treated=0)']}.

Temozolomide chemotherapy:
{row['Chemo_status (TMZ treated=1;un-treated=0)']}.

Overall survival:
{row['OS']} days.
"""

    documents.append(doc)

    metadata.append(
    {
        "patient_id": row["CGGA_ID"],
        "age": row["Age"],
        "gender": row["Gender"],
        "histology": row["Histology"],
        "grade": row["Grade"],
        "os_days": row["OS"],
        "idh": row["IDH_mutation_status"],
        "mgmt": row["MGMTp_methylation_status"],
        "1p19q": row["1p19q_codeletion_status"]
    })

print("Documents:", len(documents))

100%|██████████| 693/693 [00:00<00:00, 8220.61it/s]

Documents: 693


In [5]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"

encoder = SentenceTransformer(
    MODEL_NAME
)

print("Encoder loaded")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7156.30it/s]


Encoder loaded


In [6]:
embeddings = encoder.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)

print(embeddings.shape)

Batches: 100%|██████████| 22/22 [00:08<00:00,  2.56it/s]

(693, 1024)


In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings
)

print(index.ntotal)

693


In [8]:
faiss.write_index(
    index,
    "cgga.index"
)

np.save(
    "embeddings.npy",
    embeddings
)

with open(
    "documents.pkl",
    "wb"
) as f:
    pickle.dump(
        documents,
        f
    )

with open(
    "metadata.pkl",
    "wb"
) as f:
    pickle.dump(
        metadata,
        f
    )

print("Saved successfully")

Saved successfully


In [9]:
index = faiss.read_index(
    "cgga.index"
)

with open(
    "documents.pkl",
    "rb"
) as f:
    documents = pickle.load(f)

with open(
    "metadata.pkl",
    "rb"
) as f:
    metadata = pickle.load(f)

In [10]:
query = """
Glioblastoma patient.

Male.
Age 58.

IDH wildtype.

MGMT methylated.

WHO grade IV.
"""

In [11]:
query_embedding = encoder.encode(
    query,
    normalize_embeddings=True
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)

In [12]:
scores, indices = index.search(
    query_embedding.reshape(1,-1),
    20
)

In [13]:
query_metadata = {
    "histology": "GBM",
    "grade": "WHO IV",
    "idh": "Wildtype",
    "mgmt": "methylated",
    "1p19q": "Non-codel",
    "age": 58,
    "gender": "Male"
}

In [14]:
def rerank_candidates(
        scores,
        indices,
        metadata,
        documents,
        query_metadata):

    reranked_results = []

    for semantic_score, idx in zip(scores[0], indices[0]):

        patient = metadata[idx]

        final_score = float(semantic_score)

        # Histology
        if patient["histology"] == query_metadata["histology"]:
            final_score += 0.30

        # Grade
        if patient["grade"] == query_metadata["grade"]:
            final_score += 0.25

        # IDH
        if patient["idh"] == query_metadata["idh"]:
            final_score += 0.25

        # MGMT
        if patient["mgmt"] == query_metadata["mgmt"]:
            final_score += 0.20

        # 1p19q
        if patient["1p19q"] == query_metadata["1p19q"]:
            final_score += 0.15

        # Age
        age_difference = abs(
            patient["age"] - query_metadata["age"]
        )

        if age_difference <= 5:
            final_score += 0.10

        elif age_difference <= 10:
            final_score += 0.05

        # Gender
        if patient["gender"] == query_metadata["gender"]:
            final_score += 0.02

        reranked_results.append(
            {
                "patient_id": patient["patient_id"],
                "semantic_score": float(semantic_score),
                "final_score": final_score,
                "metadata": patient,
                "document": documents[idx]
            }
        )

    reranked_results.sort(
        key=lambda x: x["final_score"],
        reverse=True
    )

    return reranked_results

In [16]:
reranked = rerank_candidates(
    scores,
    indices,
    metadata,
    documents,
    query_metadata
)

for rank, result in enumerate(reranked[:5], start=1):

    print("="*80)

    print("Rank:", rank)

    print(
        "Semantic score:",
        result["semantic_score"]
    )

    print(
        "Final score:",
        result["final_score"]
    )

    print("\nMetadata:")
    print(result["metadata"])

    print("\nPatient Document:")
    print(result["document"])

Rank: 1
Semantic score: 0.8268297910690308
Final score: 2.096829791069031

Metadata:
{'patient_id': 'CGGA_1901', 'age': 60.0, 'gender': 'Male', 'histology': 'GBM', 'grade': 'WHO IV', 'os_days': 540.0, 'idh': 'Wildtype', 'mgmt': 'methylated', '1p19q': 'Non-codel'}

Patient Document:

Patient ID CGGA_1901.

Primary GBM.
WHO IV.

Gender: Male.
Age: 60.0 years.

IDH mutation status:
Wildtype.

1p19q codeletion status:
Non-codel.

MGMT methylation status:
methylated.

Radiotherapy:
1.0.

Temozolomide chemotherapy:
1.0.

Overall survival:
540.0 days.

Rank: 2
Semantic score: 0.8258270621299744
Final score: 2.0958270621299744

Metadata:
{'patient_id': 'CGGA_1713', 'age': 62.0, 'gender': 'Male', 'histology': 'GBM', 'grade': 'WHO IV', 'os_days': 332.0, 'idh': 'Wildtype', 'mgmt': 'methylated', '1p19q': 'Non-codel'}

Patient Document:

Patient ID CGGA_1713.

Primary GBM.
WHO IV.

Gender: Male.
Age: 62.0 years.

IDH mutation status:
Wildtype.

1p19q codeletion status:
Non-codel.

MGMT methylation 